In [3]:
from pyspark.sql import SparkSession
import csv

In [2]:
spark= SparkSession.builder.appName("Assin-app").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/16 17:02:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/16 17:02:47 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [4]:
data = [
    [1, "Amit", "IT", 55000],
    [2, "Rahul", "HR", 40000],
    [3, "Neha", "IT", 65000],
    [4, "Priya", "Finance", 70000],
    [5, "Karan", "IT", 50000],
    [6, "Simran", "HR", 45000],
    [7, "Rohit", "Finance", 60000]
]

In [6]:
with open("employee.csv","w",newline="") as dataFile:
    writer=csv.writer(dataFile)
    writer.writerow(["id","name","department","salary"])
    writer.writerows(data)

In [11]:
rdd=spark.sparkContext.textFile("/workspace/employee.csv")

In [8]:
rdd

empployee.csv MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0

In [12]:
print(rdd)

/workspace/employee.csv MapPartitionsRDD[3] at textFile at NativeMethodAccessorImpl.java:0


In [13]:
print(rdd.collect())

['id,name,department,salary', '1,Amit,IT,55000', '2,Rahul,HR,40000', '3,Neha,IT,65000', '4,Priya,Finance,70000', '5,Karan,IT,50000', '6,Simran,HR,45000', '7,Rohit,Finance,60000']


In [14]:
header = rdd.first()
data = rdd.filter(lambda row: row != header)

In [15]:
data


PythonRDD[5] at RDD at PythonRDD.scala:53

In [16]:
data.collect()

['1,Amit,IT,55000',
 '2,Rahul,HR,40000',
 '3,Neha,IT,65000',
 '4,Priya,Finance,70000',
 '5,Karan,IT,50000',
 '6,Simran,HR,45000',
 '7,Rohit,Finance,60000']

In [17]:
employees = data.map(
    lambda row: (
        int(row.split(",")[0]),   
        row.split(",")[1], 
        row.split(",")[2],       
        int(row.split(",")[3])
    )
)


In [18]:
employees.collect()

[(1, 'Amit', 'IT', 55000),
 (2, 'Rahul', 'HR', 40000),
 (3, 'Neha', 'IT', 65000),
 (4, 'Priya', 'Finance', 70000),
 (5, 'Karan', 'IT', 50000),
 (6, 'Simran', 'HR', 45000),
 (7, 'Rohit', 'Finance', 60000)]

In [22]:
sorted_employee=employees.sortBy(
    lambda emp:emp[3],
    ascending=False
)

In [24]:
print(sorted_employee.collect())

[Stage 6:>                                                          (0 + 2) / 2]

[(4, 'Priya', 'Finance', 70000), (3, 'Neha', 'IT', 65000), (7, 'Rohit', 'Finance', 60000), (1, 'Amit', 'IT', 55000), (5, 'Karan', 'IT', 50000), (6, 'Simran', 'HR', 45000), (2, 'Rahul', 'HR', 40000)]


In [27]:
# to store the sorting result in the result tables
with open("Sorting.csv","w",newline="")as sortfile:
    writer=csv.writer(sortfile)
    writer.writerow(["id","name","department","salary"])
    writer.writerows(sorted_employee.collect())

In [28]:
dept_totals = employees \
    .map(lambda emp: (emp[2], emp[3])) \
    .reduceByKey(lambda a, b: a + b)

In [33]:
with open("Total.csv","w",newline="") as totalfile:
    writer=csv.writer(totalfile)
    writer.writerow(["id","name","department","salary"])
    writer.writerow(dept_totals.collect())